# Notebook 1 - Data Loading and Cleaning

**Goal** - load raw KKBox data from 5 source files and produce one clean subscriber dataset.

All other notebooks read from `data/processed/master.csv`. This is the only notebook that touches raw files.

## Setup

In [2]:
import pandas as pd
import numpy as np
import gc
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")
#import of libraries

## File paths

In [3]:
Raw = Path("../data/raw")
MEMBERS_PATH = Raw / "members_v3.csv"
TRAIN1_PATH = Raw / "train.csv"
TRAIN2_PATH = Raw / "train_v2.csv"
TRANSACTIONS_PATH1 = Raw / "transactions.csv"
TRANSACTIONS_PATH2 = Raw / "transactions_v2.csv"
USER_LOGS_PATH = Raw / "user_logs.csv"
USER_LOGS_V2_PATH = Raw / "user_logs_v2.csv"
TRAIN_PATHS = [TRAIN1_PATH, TRAIN2_PATH]
TRANSACTIONS_PATHS = [TRANSACTIONS_PATH1, TRANSACTIONS_PATH2]

REFERENCE_DATE   = pd.Timestamp("2017-04-01")
CHUNK_SIZE       = 2_000_000
SCALE_POS_WEIGHT = 10
EXPECTED_USERS   = 1_082_190
EXPECTED_CHURN   = 0.0915

PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

train1 = pd.read_csv(TRAIN1_PATH)
train2 = pd.read_csv(TRAIN2_PATH)
train = pd.concat([train1, train2], ignore_index=True).drop_duplicates('msno').reset_index(drop=True)
print(f"Train after dedup: {train.shape[0]:,} users, churn rate: {train['is_churn'].mean():.2%}")

transactions_old = pd.read_csv(TRANSACTIONS_PATH1)
transactions_new = pd.read_csv(TRANSACTIONS_PATH2)
transactions = pd.concat([transactions_old, transactions_new], ignore_index=True)
del transactions_old, transactions_new
gc.collect()
print(f"Transactions loaded: {transactions.shape} — staging copies freed")

Train after dedup: 1,082,190 users, churn rate: 9.15%
Transactions loaded: (22978755, 9) — staging copies freed


## Check all source files exist before doing anything

In [4]:
all_paths = [MEMBERS_PATH, TRAIN1_PATH, TRAIN2_PATH,
             TRANSACTIONS_PATH1, TRANSACTIONS_PATH2,
             USER_LOGS_PATH, USER_LOGS_V2_PATH]
for p in all_paths:
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"{status}  {p.name}")

✓  members_v3.csv
✓  train.csv
✓  train_v2.csv
✓  transactions.csv
✓  transactions_v2.csv
✓  user_logs.csv
✓  user_logs_v2.csv


## Load members — registration dates and demographics

In [5]:
members = pd.read_csv(MEMBERS_PATH)

members['registration_init_time'] = pd.to_datetime(
    members['registration_init_time'].astype(str),
    format='%Y%m%d',
    errors='coerce'
)

birth_valid = members['bd'].between(1940, 2006)
members.loc[~birth_valid, 'bd'] = np.nan

print(f"Members loaded: {members.shape}")
print(f"registration_init_time dtype: {members['registration_init_time'].dtype}")
print(f"bd: {birth_valid.sum():,} valid, {(~birth_valid).sum():,} set to NaN")

Members loaded: (6769473, 6)
registration_init_time dtype: datetime64[ns]
bd: 6 valid, 6,769,467 set to NaN


## Build a registration date lookup for the log files

In [6]:
reg_dates = members.set_index('msno')['registration_init_time'].to_dict()
reg_dates_s = pd.Series(reg_dates, dtype='datetime64[ns]')
print(f"Registration lookup built: {len(reg_dates_s):,} users")

Registration lookup built: 6,769,473 users


## Aggregate transaction features

One row per subscriber. Counts, amounts, plan duration, auto-renewal history, cancellations.

In [7]:
customer_txn_count = (
    transactions.groupby("msno")
    .size()
    .reset_index(name="txn_count")
)

In [8]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"], format="%Y%m%d", errors="coerce")
transactions["membership_expire_date"] = pd.to_datetime(transactions["membership_expire_date"], format="%Y%m%d", errors="coerce")

In [9]:
LP = (
    transactions.groupby("msno")["transaction_date"].max().reset_index(name="last_transaction_date")
)

In [10]:
LTV = (
    transactions.groupby("msno")["actual_amount_paid"]
    .sum()
    .reset_index(name="total_paid")
)  

In [11]:
avg_duration = (
    transactions.groupby("msno")["payment_plan_days"]
    .mean()
    .reset_index(name="avg_plan_duration")
)  

In [12]:
auto_renew_history= (
    transactions.sort_values("transaction_date")
    .groupby("msno")["is_auto_renew"]
    .last()
    .reset_index(name="auto_renew_history")
)

In [13]:
cancel_count = (
    transactions.groupby("msno")["is_cancel"]
    .sum()
    .reset_index(name="cancel_count")
)
cancel_count.head()
cancel_count.shape
cancel_count.info()
cancel_count["cancel_count"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2426143 entries, 0 to 2426142
Data columns (total 2 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   msno          object
 1   cancel_count  int64 
dtypes: int64(1), object(1)
memory usage: 37.0+ MB


count    2.426143e+06
mean     3.676552e-01
std      5.772337e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      2.100000e+01
Name: cancel_count, dtype: float64

In [14]:
last_expire = (
    transactions.groupby("msno")["membership_expire_date"]
    .max()
    .reset_index(name="last_expire_date")
)

## Merge all transaction features into one table

In [15]:
transactions_features = LP
transactions_features = transactions_features.merge(LTV, on="msno", how="left")
transactions_features = transactions_features.merge(avg_duration, on="msno", how="left")
transactions_features = transactions_features.merge(auto_renew_history, on="msno", how="left")
transactions_features = transactions_features.merge(cancel_count, on="msno", how="left")
transactions_features = transactions_features.merge(last_expire, on="msno", how="left")
transactions_features = transactions_features.merge(customer_txn_count, on="msno", how="left")

assert transactions_features['msno'].nunique() == transactions_features.shape[0], "Duplicate msno — merge went wrong"
assert transactions_features.shape[0] == 2_426_143, f"Wrong row count: {transactions_features.shape[0]}"
print(f"✓ transactions_features ready: {transactions_features.shape}")
print(transactions_features.dtypes)

✓ transactions_features ready: (2426143, 8)
msno                             object
last_transaction_date    datetime64[ns]
total_paid                        int64
avg_plan_duration               float64
auto_renew_history                int64
cancel_count                      int64
last_expire_date         datetime64[ns]
txn_count                         int64
dtype: object


## Process user logs in chunks

Log files are ~2GB each. Loaded and aggregated in 2M-row chunks to stay within memory.

In [16]:
chunk_accum = []
total_chunks = 0

for filepath in [USER_LOGS_PATH, USER_LOGS_V2_PATH]:
    print(f"\nProcessing: {filepath.name}")
    for chunk in pd.read_csv(filepath, chunksize=CHUNK_SIZE):
        total_chunks += 1

        # Date parsing
        chunk['date'] = pd.to_datetime(
            chunk['date'].astype(str), format='%Y%m%d', errors='coerce'
        )
        chunk = chunk.dropna(subset=['date'])

        # Cap impossible values
        chunk['total_secs'] = chunk['total_secs'].clip(lower=0, upper=86_400)
        chunk['num_unq']    = chunk['num_unq'].clip(lower=0, upper=10_000)

        # Completion rate
        plays = chunk[['num_25','num_50','num_75','num_985','num_100']].sum(axis=1)
        plays = plays.replace(0, np.nan)
        chunk['completion_rate'] = (chunk['num_100'] / plays).fillna(0.0)

        # First-week tag
        chunk['reg_date']       = chunk['msno'].map(reg_dates_s)
        chunk['days_since_reg'] = (chunk['date'] - chunk['reg_date']).dt.days
        is_first_week           = chunk['days_since_reg'].between(0, 6)

        # Aggregate all rows
        agg_all = chunk.groupby('msno', sort=False).agg(
            date_max       = ('date',            'max'),
            date_count     = ('date',            'count'),
            num_unq_sum    = ('num_unq',         'sum'),
            total_secs_sum = ('total_secs',      'sum'),
            completion_sum = ('completion_rate', 'sum'),
        ).reset_index()

        # Aggregate first-week rows
        fw = chunk[is_first_week].groupby('msno', sort=False).agg(
            fw_num_unq_sum = ('num_unq', 'sum'),
            fw_day_count   = ('date',    'count'),
        ).reset_index()

        agg_all = agg_all.merge(fw, on='msno', how='left')
        agg_all['fw_num_unq_sum'] = agg_all['fw_num_unq_sum'].fillna(0)
        agg_all['fw_day_count']   = agg_all['fw_day_count'].fillna(0)

        chunk_accum.append(agg_all)

        if total_chunks % 5 == 0:
            print(f"  Chunk {total_chunks} done")

print(f"\nTotal chunks processed: {total_chunks}")


Processing: user_logs.csv
  Chunk 5 done
  Chunk 10 done
  Chunk 15 done
  Chunk 20 done
  Chunk 25 done
  Chunk 30 done
  Chunk 35 done
  Chunk 40 done
  Chunk 45 done
  Chunk 50 done
  Chunk 55 done
  Chunk 60 done
  Chunk 65 done
  Chunk 70 done
  Chunk 75 done
  Chunk 80 done
  Chunk 85 done
  Chunk 90 done
  Chunk 95 done
  Chunk 100 done
  Chunk 105 done
  Chunk 110 done
  Chunk 115 done
  Chunk 120 done
  Chunk 125 done
  Chunk 130 done
  Chunk 135 done
  Chunk 140 done
  Chunk 145 done
  Chunk 150 done
  Chunk 155 done
  Chunk 160 done
  Chunk 165 done
  Chunk 170 done
  Chunk 175 done
  Chunk 180 done
  Chunk 185 done
  Chunk 190 done
  Chunk 195 done

Processing: user_logs_v2.csv
  Chunk 200 done
  Chunk 205 done

Total chunks processed: 207


## Collapse chunks — one row per subscriber

In [17]:
print("combining chunks")
ul_raw = pd.concat(chunk_accum, ignore_index=True)
del chunk_accum
gc.collect()
#collapse to one row per user
ul_agg = ul_raw.groupby('msno', sort=False).agg(
    last_active_date = ('date_max', 'max'),
    activity_day_count = ('date_count', 'sum'),
    num_unq_sum = ('num_unq_sum', 'sum'),
    total_secs_sum = ('total_secs_sum', 'sum'),
    completion_sum = ('completion_sum', 'sum'),
    fw_num_unq_sum = ('fw_num_unq_sum', 'sum'),
    fw_day_count = ('fw_day_count', 'sum'),
).reset_index()
del ul_raw
gc.collect()
#compute metrics
ul_agg['avg_num_unq']         = (ul_agg['num_unq_sum']   / ul_agg['activity_day_count']).round(2)
ul_agg['avg_total_secs']      = (ul_agg['total_secs_sum'] / ul_agg['activity_day_count']).round(1)
ul_agg['avg_completion_rate'] = (ul_agg['completion_sum'] / ul_agg['activity_day_count']).round(4)
ul_agg['first_week_depth'] = (
    ul_agg['fw_num_unq_sum'] / ul_agg['fw_day_count'].replace(0, np.nan)
).fillna(0).round(2)
assert ul_agg['msno'].nunique() == ul_agg.shape[0],"Duplicate msno in ul_agg"
assert ul_agg['avg_total_secs'].max() <= 86_400, "clipping err"
print(f"ul_agg ready: {ul_agg.shape}")
print(ul_agg[['avg_num_unq', 'avg_total_secs', 'avg_completion_rate', 'first_week_depth']].describe())

combining chunks
ul_agg ready: (5339422, 12)
        avg_num_unq  avg_total_secs  avg_completion_rate  first_week_depth
count  5.339422e+06    5.339422e+06         5.339422e+06      5.339422e+06
mean   1.676728e+01    3.867383e+03         4.800362e-01      1.136571e+01
std    1.821465e+01    4.899894e+03         3.160181e-01      1.866114e+01
min    1.000000e+00    0.000000e+00         0.000000e+00      0.000000e+00
25%    4.090000e+00    6.246000e+02         1.989000e-01      0.000000e+00
50%    1.182000e+01    2.362100e+03         5.274000e-01      4.000000e+00
75%    2.300000e+01    5.282700e+03         7.420000e-01      1.514000e+01
max    1.402720e+03    8.640000e+04         1.000000e+00      8.530000e+02


## Build master dataset

In [18]:
#
#
master = train.copy()
before = master.shape[0]
master = master.merge(members, on='msno', how='left')
assert master.shape[0] == before, "Members join changed row count"
master = master.merge(transactions_features, on='msno', how='left')
assert master.shape[0] == before, "Transactions join changed row count"
master = master.merge(
    ul_agg[['msno', 'last_active_date', 'activity_day_count',
            'avg_num_unq', 'avg_total_secs', 'avg_completion_rate', 'first_week_depth']],
    on='msno', how='left'
)
assert master.shape[0] == before, "User logs join changed row count"
#flag missing
master['has_tx'] = master['total_paid'].notna().astype(int)
master['has_logs'] = master['avg_num_unq'].notna().astype(int)
#derived time features
master['tenure_days'] = (REFERENCE_DATE - master['registration_init_time']).dt.days.clip(lower=0)
master['days_since_last_active'] = (REFERENCE_DATE - master['last_active_date']).dt.days
master['revenue_per_day'] = (
    master['total_paid'] / master['tenure_days'].replace(0, np.nan)
).round(4)
print(f"Master dataset ready: {master.shape}")
print(f"has_tx=1: {master['has_tx'].sum():,} users have transaction data")
print(f"has_logs=1: {master['has_logs'].sum():,} users have log data")

Master dataset ready: (1082190, 25)
has_tx=1: 1,082,190 users have transaction data
has_logs=1: 955,159 users have log data


## Validate output

In [19]:
print("=" * 50)
print("Data Pipeline Initialized")
print("=" * 50)
#Row count check
assert master.shape[0] == EXPECTED_USERS, \
    f"FAIL: {master.shape[0]:,} rows is not equal to expected {EXPECTED_USERS:,}"
print(f"check Row count: {master.shape[0]:,}")

#every id must be unique
assert master['msno'].nunique() == master.shape[0], \
    f"FAIL: Duplicate msno found"
print(f"correct, every msno is unique: {master['msno'].nunique():,}")
#churn rate check (must be within 0.2% of expected)
assert abs(master['is_churn'].mean() - EXPECTED_CHURN) < 0.002, \
    f"FAIL: Churn rate {master['is_churn'].mean():.4%} is not within 0.2% of expected {EXPECTED_CHURN:.4%}"
print(f"correct, churn rate is within 0.2% of expected: {master['is_churn'].mean():.2%}")

#Check of negative tenure_days
assert (master['tenure_days'].fillna(0) >= 0).all(), \
    f"FAIL: Negative tenure_days found"
print(f"correct, no negative tenure_days found")

#null rates
null_rates = master.isnull().mean().sort_values(ascending=False)
for col, rate in null_rates[null_rates > 0].items():
    print(f"WARNING: {col} has {rate:.1%} ")

print("Data pipeline completed successfully.")

Data Pipeline Initialized
check Row count: 1,082,190
correct, every msno is unique: 1,082,190
correct, churn rate is within 0.2% of expected: 9.15%
correct, no negative tenure_days found
Data pipeline completed successfully.


## Save

In [20]:
MASTER_PATH = PROCESSED / "master.csv"
master.to_csv(MASTER_PATH, index=False)
verify = pd.read_csv(MASTER_PATH, usecols=[ 'msno', 'is_churn' ])
assert verify.shape[0] == EXPECTED_USERS, "save verification failed"
print(f"master.csv saved")
print(f"Rows: {master.shape[0]:,}, Columns: {master.shape[1]:,}")
print(f" Size: {MASTER_PATH.stat().st_size / 1e6:.1f} MB  ")
print(f"{MASTER_PATH}")

master.csv saved
Rows: 1,082,190, Columns: 25
 Size: 182.9 MB  
..\data\processed\master.csv


## Conclusion

1,082,190 subscribers loaded and validated. Churn rate 9.15%. Master file saved to `data/processed/master.csv`.

Next: notebook 2 explores what separates churners from retained subscribers.